# 05 — RGCN Model (Relation-Aware GCN) & Comparison

Extends the pipeline with a **Relational GCN** (`RGCNConv`, Schlichtkrull et al. 2018) trained on the same
graph built in [03_preprocessing_gnn_features.ipynb](03_preprocessing_gnn_features.ipynb), evaluated on the
same time-based validation split as every other notebook here.

**Why this notebook exists.** [04_gcn_model.ipynb](04_gcn_model.ipynb) scored a disappointing 0.60 val AUC
vs. 0.80 for the tabular baseline. Digging into why (see the discussion at the end of notebook 04) surfaced
two concrete issues with a plain `GCNConv`:

1. **`edge_type` is computed by notebook 03 but never used by notebook 04.** `GCNConv` treats every edge —
   whether it's "shares a rare device" (strong signal) or "shares `card3`" (only ~490 edges total, close to
   noise) — with the same symmetric-normalized weight.
2. **The graph mixes 12 structurally very different relations** (`card1`–`card6`, `addr1`–`addr2`,
   `P_emaildomain`, `R_emaildomain`, `DeviceType`, `DeviceInfo`) into one flat adjacency. `card1` alone
   accounts for ~77% of all edges, so a relation-blind conv effectively lets one weak-ish relation dominate
   message passing.

`RGCNConv` addresses exactly this: it learns a **separate weight matrix per relation**, aggregates neighbors
relation-by-relation, then sums the per-relation messages at each node — so the model can learn to trust
`DeviceInfo`/email-match edges more than `card3`/`card4` edges, instead of averaging them all together. This
mirrors the reference architecture in the AWS `sagemaker-graph-fraud-detection` DGL example (`HeteroRGCNLayer`)
that the Kaggle RGCN notebook is built on.

Requires: `pip install torch torch-geometric`.


In [ ]:
import sys, pathlib, json, time
sys.path.append(str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fraud_utils import (PROCESSED_DIR, RESULTS_DIR, RANDOM_SEED, set_seed,
                          save_metrics, load_metrics, save_predictions, load_predictions)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.data import Data
    from torch_geometric.nn import RGCNConv
except ImportError as e:
    raise ImportError(
        "This notebook needs torch + torch-geometric. Install with:\n"
        "  pip install torch torchvision\n"
        "  pip install torch-geometric\n"
        "(see https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html "
        "for platform-specific wheels)."
    ) from e

from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                              precision_recall_curve, confusion_matrix, classification_report)

set_seed()
plt.rcParams["figure.figsize"] = (7, 5)
%matplotlib inline

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")  # Apple Silicon
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")


## 1. Load Preprocessed Graph Artifacts (Including Edge Types)

Unlike notebook 04, we actually attach `edge_type` to the PyG `Data` object and thread it through every
model call below — that's the whole point of this notebook.


In [ ]:
features = np.load(PROCESSED_DIR / "features.npy")
labels = np.load(PROCESSED_DIR / "labels.npy")
split_arr = np.load(PROCESSED_DIR / "split.npy", allow_pickle=True)
edge_index_np = np.load(PROCESSED_DIR / "edge_index.npy")
edge_type_np = np.load(PROCESSED_DIR / "edge_type.npy")
with open(PROCESSED_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

# EDGE_TYPE_COLS from 03_preprocessing_gnn_features.ipynb, for readable relation names below
EDGE_TYPE_COLS = ["card1", "card2", "card3", "card4", "card5", "card6",
                   "addr1", "addr2", "P_emaildomain", "R_emaildomain",
                   "DeviceType", "DeviceInfo"]

NUM_RELATIONS = int(edge_type_np.max()) + 1
print(f"features: {features.shape}, edges: {edge_index_np.shape[1]:,}, "
      f"n_features: {len(feature_names)}, n_relations: {NUM_RELATIONS}")

rel_counts = pd.Series(edge_type_np).value_counts().sort_index()
for rel_idx, count in rel_counts.items():
    name = EDGE_TYPE_COLS[rel_idx] if rel_idx < len(EDGE_TYPE_COLS) else f"relation_{rel_idx}"
    print(f"  relation {rel_idx:2d} ({name:16s}): {count:>10,d} edges")

x = torch.tensor(features, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.float32)
edge_index = torch.tensor(edge_index_np, dtype=torch.long)
edge_type = torch.tensor(edge_type_np, dtype=torch.long)

train_mask = torch.tensor(split_arr == "train", dtype=torch.bool)
valid_mask = torch.tensor(split_arr == "valid", dtype=torch.bool)
test_mask = torch.tensor(split_arr == "test", dtype=torch.bool)

data = Data(x=x, y=y, edge_index=edge_index, edge_type=edge_type,
            train_mask=train_mask, valid_mask=valid_mask, test_mask=test_mask)
print(data)
print(f"train: {train_mask.sum().item():,}  valid: {valid_mask.sum().item():,}  test: {test_mask.sum().item():,}")


## 2. RGCN Model Definition

Same overall shape as `FraudGCN` in [04_gcn_model.ipynb](04_gcn_model.ipynb) (input projection → stack of
conv layers with BatchNorm + ReLU + Dropout → concatenate every layer's output as skip connections → MLP
classifier head), but each conv layer is now `RGCNConv`, which:

- Learns one weight matrix `W_r` per relation `r` and aggregates neighbors **within** each relation before
  summing **across** relations — the model can learn to lean on `DeviceInfo`/email-match edges and
  down-weight noisy ones like `card3`/`card4`/`card6`.
- Has a built-in root/self transformation (`root_weight=True`, PyG default), so it doesn't need an explicit
  self-loop relation the way the flat `GCNConv` graph implicitly does.

**`num_bases`**: with 12 relations, several of them (`card3`, `card4`, `card6`, `DeviceType`) have only a few
hundred edges total — not much data to learn a full independent 128×128 weight matrix per relation from
scratch. Basis-decomposition regularization (`num_bases`) makes every relation's weight matrix a learned
linear combination of a small shared set of `num_bases` basis matrices, so rare relations borrow statistical
strength from common ones instead of overfitting/underfitting in isolation. Set `NUM_BASES = None` below to
disable it and give every relation a fully independent weight matrix instead.


In [ ]:
class FraudRGCN(nn.Module):
    def __init__(self, in_channels, num_relations, hidden_channels=128, num_layers=3,
                 dropout=0.3, num_bases=None):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout

        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.input_bn = nn.BatchNorm1d(hidden_channels)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(RGCNConv(hidden_channels, hidden_channels, num_relations=num_relations,
                                        num_bases=num_bases))
            self.bns.append(nn.BatchNorm1d(hidden_channels))

        classifier_in = hidden_channels * (num_layers + 1)  # skip-connected concat
        self.classifier = nn.Sequential(
            nn.Linear(classifier_in, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.BatchNorm1d(hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(self, x, edge_index, edge_type):
        h = F.relu(self.input_bn(self.input_proj(x)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        skip = [h]
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index, edge_type)
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            skip.append(h)
        h = torch.cat(skip, dim=-1)
        return self.classifier(h).squeeze(-1)


## 3. Training Setup

Full-batch training, identical hyperparameters to notebook 04 where possible so the comparison isolates the
architecture change (`GCNConv` → `RGCNConv` + relation awareness) rather than a hyperparameter change. If you
hit an out-of-memory error, swap this loop for PyG's `NeighborLoader` mini-batching, as `../gnn_fraud_detection.py`
does — the model class above is unchanged either way.


In [ ]:
HIDDEN_DIM = 128
NUM_LAYERS = 3
DROPOUT = 0.3
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 60
NUM_BASES = 6  # basis-decomposition regularization; set to None for a full independent weight per relation

model = FraudRGCN(in_channels=x.shape[1], num_relations=NUM_RELATIONS, hidden_channels=HIDDEN_DIM,
                   num_layers=NUM_LAYERS, dropout=DROPOUT, num_bases=NUM_BASES).to(DEVICE)
data = data.to(DEVICE)

n_pos = (data.y[data.train_mask] == 1).sum()
n_neg = (data.y[data.train_mask] == 0).sum()
pos_weight = (n_neg / n_pos).clamp(min=1.0)
print(f"pos_weight (class imbalance correction): {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

CKPT_PATH = RESULTS_DIR / "best_rgcn_model.pt"


In [ ]:
@torch.no_grad()
def evaluate(mask):
    model.eval()
    out = model(data.x, data.edge_index, data.edge_type)
    probs = torch.sigmoid(out[mask])
    y_true = data.y[mask].cpu().numpy()
    y_score = probs.cpu().numpy()
    auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float("nan")
    return auc, y_true, y_score


best_val_auc = 0.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.edge_type)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    val_auc, _, _ = evaluate(data.valid_mask)
    history.append({"epoch": epoch, "loss": loss.item(), "val_auc": val_auc})

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | loss {loss.item():.4f} | val AUC {val_auc:.4f} | "
              f"lr {scheduler.get_last_lr()[0]:.6f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), CKPT_PATH)

model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
print(f"\nBest validation AUC: {best_val_auc:.4f}")


In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_df["epoch"], hist_df["loss"], color="#4C72B0")
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch")
axes[1].plot(hist_df["epoch"], hist_df["val_auc"], color="#C44E52")
axes[1].set_title("Validation AUC"); axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()


## 4. Evaluate on Validation Set

In [ ]:
rgcn_auc, y_valid, rgcn_valid_scores = evaluate(data.valid_mask)
rgcn_ap = average_precision_score(y_valid, rgcn_valid_scores)
print(f"RGCN  |  valid AUC: {rgcn_auc:.4f}  |  valid PR-AUC: {rgcn_ap:.4f}")

print("\nClassification report @ threshold 0.5:")
print(classification_report(y_valid, (rgcn_valid_scores > 0.5).astype(int), digits=4))
print(confusion_matrix(y_valid, (rgcn_valid_scores > 0.5).astype(int)))


## 5. Which Relations Does the Model Actually Lean On?

A direct payoff of relation-specific weights: we can inspect how much each relation's weight matrix
contributes at the first `RGCNConv` layer. A relation whose weight matrix has near-zero norm is being
effectively ignored by the model — useful for checking whether it learned to down-weight the noisy
`card3`/`card4`/`card6` relations we flagged in notebook 04's discussion. This only works directly when
`NUM_BASES is None` (a genuine per-relation weight matrix); with basis decomposition the effective per-relation
weight is a learned combination of shared bases, so we reconstruct it from `comp`/`basis` instead.


In [ ]:
first_conv = model.convs[0]
with torch.no_grad():
    if first_conv.num_bases is not None:
        # effective per-relation weight = comp @ basis, reshaped to (num_relations, in, out)
        rel_weight = torch.einsum("rb,bio->rio", first_conv.comp, first_conv.basis)
    else:
        rel_weight = first_conv.weight
    rel_norms = rel_weight.flatten(1).norm(dim=1).cpu().numpy()

rel_importance = pd.DataFrame({
    "relation": [EDGE_TYPE_COLS[i] if i < len(EDGE_TYPE_COLS) else f"relation_{i}" for i in range(NUM_RELATIONS)],
    "edge_count": [rel_counts.get(i, 0) for i in range(NUM_RELATIONS)],
    "layer0_weight_norm": rel_norms,
}).sort_values("layer0_weight_norm", ascending=False)
rel_importance


## 6. Save RGCN Metrics

In [ ]:
rgcn_metrics = {
    "model": "RGCN",
    "auc": float(rgcn_auc),
    "pr_auc": float(rgcn_ap),
    "n_train": int(data.train_mask.sum().item()),
    "n_valid": int(data.valid_mask.sum().item()),
    "n_features": len(feature_names),
    "n_relations": NUM_RELATIONS,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "num_bases": NUM_BASES,
    "num_epochs_trained": len(history),
}
print("--- rgcn metrics (about to save) ---")
print(json.dumps(rgcn_metrics, indent=2, default=float))
save_metrics("rgcn", rgcn_metrics)
save_predictions("rgcn", y_valid, rgcn_valid_scores)

## 7. Compare vs. Baseline & Plain GCN

Loads every metrics/predictions file saved so far. All models were evaluated on the **identical** validation
rows (same `TransactionDT`-based split), so AUC/PR-AUC are directly comparable.


In [ ]:
model_names = ["baseline_logreg", "baseline_lightgbm", "gcn_gcn", "rgcn"]
rows = []
for name in model_names:
    try:
        m = load_metrics(name)
        rows.append({"notebook": name, **m})
    except FileNotFoundError:
        print(f"(skipping {name}: metrics file not found — run its notebook first)")

comparison = pd.DataFrame(rows).set_index("notebook")[["model", "auc", "pr_auc", "n_train", "n_valid", "n_features"]]
comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
comparison["auc"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Validation AUC"); axes[0].set_ylim(0.5, 1.0); axes[0].tick_params(axis="x", rotation=30)
comparison["pr_auc"].plot(kind="bar", ax=axes[1], color="#55A868")
axes[1].set_title("Validation PR-AUC"); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name in model_names:
    try:
        yt, ys = load_predictions(name)
    except FileNotFoundError:
        continue
    label = load_metrics(name)["model"]
    fpr, tpr, _ = roc_curve(yt, ys)
    axes[0].plot(fpr, tpr, label=f"{label} (AUC={roc_auc_score(yt, ys):.3f})")
    prec, rec, _ = precision_recall_curve(yt, ys)
    axes[1].plot(rec, prec, label=f"{label} (AP={average_precision_score(yt, ys):.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC — all models"); axes[0].legend()
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("PR — all models"); axes[1].legend()
plt.tight_layout()
plt.show()


## 8. Discussion

- **Did relation-awareness close the gap to the plain GCN (and to the tabular baseline)?** Check the table/plots
  above. If `RGCN` clears `gcn_gcn` by a wide margin, that confirms the diagnosis from notebook 04: the flat,
  relation-blind graph was the bottleneck, not message passing itself.
- **Section 5's relation weight norms** tell you *which* edges the model actually found useful. If
  `card3`/`card4`/`card6`/`DeviceType` end up with the smallest norms, that matches the edge-count-based
  suspicion from notebook 04 that these relations carry little signal — and it's now something the model
  discovered rather than something we assumed.
- **Still expect a real ceiling from the underlying graph construction**, independent of architecture:
  [03_preprocessing_gnn_features.ipynb](03_preprocessing_gnn_features.ipynb) caps shared-attribute groups at
  `MAX_GROUP_SIZE=50` and folds large groups into a single-hub star topology, which leaves the graph very
  sparse (avg degree ~1.1 across ~1.1M nodes). Relation-specific weights help RGCN make the *most* of that
  sparse structure, but they can't manufacture edges that were capped away. If RGCN still trails the tabular
  baseline by a lot, the next lever is graph density/construction (bigger `MAX_GROUP_SIZE`, or a true bipartite
  transaction↔attribute heterograph like the AWS/Kaggle reference, instead of folding attributes into
  transaction–transaction cliques/stars) rather than the conv operator.
- **Ideas beyond this notebook** (see [../README.md](../README.md) "Potential Improvements"):
  1. Ensemble: concatenate RGCN node embeddings onto the notebook-2 LightGBM feature set
  2. Temporal edges: only connect transactions close in time, not just sharing an attribute
  3. A true heterogeneous graph (`HeteroConv` / bipartite transaction↔attribute nodes) instead of folding
     attributes into transaction–transaction edges
  4. Try `GATConv`/`TransformerConv` for learned per-edge attention on top of (or instead of) relation labels
